<a href="https://colab.research.google.com/github/ssykes-eth/ETH_275-0005-00L/blob/code_exercises/06_cx_rag_evaluation_student.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Evaluating a RAG Assistant: Finding Out *Which* Part Is Broken

**A hands-on companion to "Evaluating Retrieval-Augmented Generation"**

## The situation

You are the product lead for the customer-facing help assistant at **Meridian Bank**. It
answers questions about accounts, cards and fees, drawing on the bank's own published
documents — product terms, quarterly fee schedules, FAQ pages. It has been in pilot with
5,000 customers. In three weeks it goes to everyone.

The thing that makes this your decision and not an engineer's: **an answer on the bank's
website is a statement by the bank.** If a customer acts on it, the bank pays. You do not
build the retriever — a vendor does. You have to decide whether it ships, and tell the
vendor precisely what to fix.

## The complaint that landed this morning

A customer asked:

> *"What will it cost me to send €2,000 to Poland?"*

The assistant answered: **€34.00**, quoting the bank's currency conversion margin of 1.7%.

The true cost is **€53.90**: that €34.00 margin, *plus* a €4.90 fixed transfer fee, *plus* a
€15.00 intermediary bank fee. Two hundred customers were quoted the low figure, and complained
when the real one reached their statements.

Now answer this: **whose fault was that?** The retrieval vendor's? The language model's?
Whoever loads the documents? You cannot tell from the answer, and that is the whole problem
this notebook solves.

Over seven parts you will build the instruments that tell you which stage broke:

1. **Five suspects, one symptom** — why the final answer never tells you what to fix.
2. **What counts as the right answer** — three ways to grade the same result, three different
   verdicts.
3. **Did we find the evidence?** — and what more evidence costs.
4. **Did we find it first?** — ranking, and what a reranker can and cannot do.
5. **Best parts ≠ best machine** — why a vendor's component benchmark is not evidence.
6. **Right is not the same as supported** — the failure where everything worked and the
   answer was still wrong.
7. **Using a model as the evaluator** — you cannot hand-grade every release forever, so can a
   model take over the grading, and how would you know whether to trust it?


In [ ]:
#@title 🗺️ Roadmap — the instruments you will build (run me) { display-mode: "form" }
import uuid
from IPython.display import HTML, display

_uid = uuid.uuid4().hex[:8]
_steps = [
    ("0", "Five suspects", "one wrong answer,<br>five possible causes"),
    ("1", "Ground truth", "what counts as<br>the right answer?"),
    ("2", "Recall", "did we find<br>the evidence?"),
    ("3", "Ranking", "did we find<br>it first?"),
    ("4", "The bake-off", "best parts ≠<br>best machine"),
    ("5", "Correct vs supported", "right, wrong,<br>and faithful"),
    ("6", "Model as evaluator", "can a model<br>do the grading?"),
]
_cards = "".join(f'''
  <div style="flex:1 1 150px;min-width:150px;background:#fff;border:1px solid #e6e8ee;
              border-radius:14px;padding:14px 12px;text-align:center;">
    <div style="width:38px;height:38px;margin:0 auto 9px;border-radius:50%;
                background:linear-gradient(135deg,#667eea,#764ba2);color:#fff;font-weight:700;
                font-size:15px;line-height:38px;">{n}</div>
    <div style="font-weight:650;font-size:13.5px;color:#22243a;margin-bottom:4px;">{t}</div>
    <div style="font-size:11.5px;color:#6b6f80;line-height:1.45;">{s}</div>
  </div>''' for n, t, s in _steps)

display(HTML(f'''
<div id="rm{_uid}" style="font-family:system-ui,Segoe UI,Roboto,sans-serif;border-radius:18px;
     border:1px solid #ecebff;background:linear-gradient(135deg,#f6f8ff,#fbf5ff);padding:20px;">
  <div style="font-size:15px;font-weight:700;color:#3c3f58;margin-bottom:14px;">
    From one wrong answer to a diagnosis</div>
  <div style="display:flex;flex-wrap:wrap;gap:10px;">{_cards}</div>
</div>'''))

In [ ]:
#@title Setup — imports, styling, palette (run me) { display-mode: "form" }
import warnings, os, json, uuid, textwrap
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import HTML, display

plt.rcParams["figure.dpi"] = 110
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False
plt.rcParams["font.size"] = 9

PURPLE, PURPLE2 = "#667eea", "#764ba2"
GREEN, RED, AMBER = "#39b36a", "#e0796d", "#e0a23c"
RANDOM_STATE = 42

pd.set_option("display.max_colwidth", 90)
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 20)
print("ready")

## Getting the data

The bank's documents and the labelled test set live in the course repo. The cell below clones
it — you need a `GITHUB_TOKEN`, either in Colab Secrets (the key icon in the left sidebar) or
as an environment variable.

Nothing else is downloaded: no model provider, no API key. Every grader's scores were computed
once, in advance, and frozen into the files.

In [ ]:
#@title Clone the course repo and load the data (run me) { display-mode: "form" }
import glob, subprocess

REPO = "eth-fdd-fs26/FDD-WE5-public"
CLONE_DIR = "FDD-WE5-public"
DATA_NAME = "06_cx_eval_data"


def _find(root="."):
    hits = glob.glob(f"{root}/**/{DATA_NAME}/chunks.parquet", recursive=True)
    return os.path.dirname(hits[0]) if hits else None


DATA_DIR = _find()          # already beside the notebook? then nothing to clone

if DATA_DIR is None:
    if not os.path.exists(CLONE_DIR):
        # -q keeps output quiet so the token is not echoed into the cell output
        subprocess.run(["git", "clone", "-q",
                        f"https://github.com/{REPO}.git",
                        CLONE_DIR], check=True)
        # scrub the token from the stored remote so it is not left on disk
        subprocess.run(["git", "-C", CLONE_DIR, "remote", "set-url", "origin",
                        f"https://github.com/{REPO}.git"], check=True)

    DATA_DIR = _find(CLONE_DIR)
    assert DATA_DIR, f"{DATA_NAME}/ not found inside {CLONE_DIR}. Check the repo layout."

print("data dir:", DATA_DIR)

chunks   = pd.read_parquet(f"{DATA_DIR}/chunks.parquet").to_dict("records")
for c in chunks:
    c["fact_ids"] = [x for x in c["fact_ids"].split("|") if x]
BY_ID    = {c["chunk_id"]: c for c in chunks}
QUESTIONS = json.load(open(f"{DATA_DIR}/eval_set.json"))
BY_QID    = {q["qid"]: q for q in QUESTIONS}
ANSWERS   = json.load(open(f"{DATA_DIR}/answers.json"))
FACTS     = json.load(open(f"{DATA_DIR}/facts.json"))
META      = json.load(open(f"{DATA_DIR}/meta.json"))
RUNS      = json.load(open(f"{DATA_DIR}/runs.json"))["runs"]

# The assistant that is live today: dense search, then a reranker over the top 50.
LIVE = RUNS["pipeline_dense_reranked"]
SCORED = [q for q in QUESTIONS if q["type"] != "unanswerable"]

print(f"{len({c['doc_id'] for c in chunks})} documents -> {len(chunks)} passages")
print(f"{len(QUESTIONS)} test questions, {len(ANSWERS)} answers to grade")
print(f"current fee schedule: {META['current_version']}   "
      f"superseded (still indexed): {META['stale_version']}")

---

# Part 1 — Five suspects, one symptom

## What the assistant is working with

Before judging anything it got wrong, look at what it has to draw on. These are the bank's
published documents — its only source. Nothing else is in the system.

In [ ]:
#@title What the bank has published (run me) { display-mode: "form" }
docs = {}
for c in chunks:
    d = docs.setdefault(c["doc_id"], {"title": c["doc_title"], "kind": c["doc_type"],
                                      "version": c["doc_version"], "passages": 0, "facts": set()})
    d["passages"] += 1
    d["facts"].update(c["fact_ids"])

KIND = {"fee_schedule": "Fee schedules", "terms": "Terms and annexes",
        "faq": "Customer FAQ pages", "policy": "Internal policies",
        "product_info": "Product info sheets"}
for kind in ["fee_schedule", "terms", "faq", "policy", "product_info"]:
    group = [d for d in docs.values() if d["kind"] == kind]
    print(f"{KIND[kind].upper()}  —  {len(group)} documents, "
          f"{sum(d['passages'] for d in group)} passages")
    for d in sorted(group, key=lambda x: x["title"])[:6]:
        v = f"  [{d['version']}]" if d["version"] else ""
        print(f"   - {d['title']}{v}")
        print(f"       {d['passages']} passages, states {len(d['facts'])} of the bank's fees")
    if len(group) > 6:
        print(f"   ... and {len(group) - 6} more of the same shape")
    print()

Two things in that list decide most of what follows.

**Every fee schedule exists twice** — the quarter in force, and the one it replaced. Both are
still in the index. Nobody removed the old one.

**There are nine near-identical "Sending money to *country*" pages.** Real banks publish one
per corridor. They differ only in the country name, which will matter a great deal.

And these are the questions we will hold it to. Each has a known correct answer, because the
documents were written *from* a single table of fees — so the answer key was computed rather
than written by hand.

In [ ]:
#@title The 25 test questions and their correct answers (run me) { display-mode: "form" }
TYPE_LABEL = {"single_fact": "one number, stated in one passage",
              "multi_fact": "needs several fees, and they live in different documents",
              "stale_trap": "last quarter's schedule is also in the index",
              "unanswerable": "the bank's documents genuinely do not cover it"}
for t, label in TYPE_LABEL.items():
    sel = [x for x in QUESTIONS if x["type"] == t]
    print(f"{t.upper().replace('_', ' ')}  ({len(sel)}) — {label}\n")
    for x in sel:
        print(f"   [{x['qid']}] {x['question']}")
        for line in textwrap.wrap(x["reference_answer"], 84):
            print(f"          {line}")
        print()

## Three complaints from this month

Now the failures. Here are three, side by side: the question, what the assistant replied, the
correct answer, and — crucially — **the passages it was working from when it replied**.

Read all three before answering anything. They are not the same problem, they do not have the
same fix, and from the customer's seat they look identical.

In [ ]:
#@title The three complaints, with the evidence each answer was given (run me) { display-mode: "form" }
for aid in ["q03_a2", "q17_a2", "q21_a2"]:
    a = next(x for x in ANSWERS if x["answer_id"] == aid)
    qq = BY_QID[a["qid"]]
    print("=" * 96)
    print(f"QUESTION:  {qq['question']}")
    print("\nIT REPLIED:")
    for line in textwrap.wrap(a["text"], 90):
        print(f"   {line}")
    print("\nTHE CORRECT ANSWER WAS:")
    for line in textwrap.wrap(qq["reference_answer"], 90):
        print(f"   {line}")
    print(f"\nIT WAS SHOWN {len(a['context_chunk_ids'])} PASSAGE(S):")
    if not a["context_chunk_ids"]:
        print("   (nothing was retrieved at all)")
    for cid in a["context_chunk_ids"]:
        p = BY_ID[cid]
        stamp = f"   [effective {p['effective_date']}]" if p["effective_date"] else ""
        print(f"   - {p['doc_title']}{stamp}")
        for line in textwrap.wrap(p["text"], 86):
            print(f"       {line}")
    print()

## Exercise 1. Same symptom, three different causes

Three wrong answers. Each broke somewhere different, and each belongs to a different team.
Work out which, from the evidence above, before revealing.

This is the exercise the whole notebook exists to replace with numbers — so form your own view
first, while you still have to read the traces by hand.

In [ ]:
#@title 🎯 Quiz — who broke each one? (run me) { display-mode: "form" }
_uid = uuid.uuid4().hex[:8]
_OPTS = ["The document was never indexed, or the indexed copy is out of date",
         "The evidence is in the index, but search did not return it",
         "Search returned it, but the reranker buried it below the cut-off",
         "The model was shown the right evidence and wrote around it",
         "The model was shown nothing relevant and answered anyway"]
QUIZ = [
    ("<b>Complaint 1 — cash withdrawal in the United States.</b> It quoted the EUR 3.50 fixed "
     "fee and stopped there. The real cost is EUR 9.50. What went wrong?", 2,
     "Every word of that answer is in the passage it was given &mdash; the card-abroad FAQ, "
     "which states the fixed fee. The missing 2% lives in a different document, the "
     "<i>Card Charges Annex</i>, and that annex is in the index. Search simply never "
     "returned it. <b>Fix belongs to: whoever tunes retrieval.</b>"),
    ("<b>Complaint 2 — the intermediary bank fee.</b> It quoted EUR 12.00 with a precise "
     "citation. The correct figure is EUR 15.00. What went wrong?", 1,
     "Look at the effective date on the passage it used: the <i>2025Q4</i> annex, replaced in "
     "January and never removed from the index. The assistant quoted it exactly and "
     "correctly. Retrieval worked, the model worked, and the answer is still wrong. "
     "<b>Fix belongs to: whoever loads and retires documents.</b>"),
    ("<b>Complaint 3 — mortgage rates.</b> It gave a confident figure with a caveat. "
     "What went wrong?", 5,
     "It was shown <i>nothing</i> &mdash; the bank publishes no mortgage documents at all, so "
     "there was nothing to retrieve. Rather than saying so, it produced a plausible number "
     "from its own training. This is the only one of the three where the model itself is at "
     "fault. <b>Fix belongs to: whoever sets the model&rsquo;s instructions.</b>"),
]
_blocks = ""
for _n, (_q, _correct, _why) in enumerate(QUIZ, 1):
    _o = "".join(f'<li style="margin:3px 0;color:#3c3f58;">{chr(64 + _i)}. {_t}</li>'
                 for _i, _t in enumerate(_OPTS, 1))
    _blocks += (
        '<div style="background:#fff;border:1px solid #e6e8ee;border-radius:14px;'
        'padding:14px 16px;margin-bottom:11px;">'
        f'<div style="font-size:13.5px;color:#22243a;margin-bottom:8px;">{_n}. {_q}</div>'
        f'<ol style="margin:0 0 9px 18px;padding:0;font-size:12.5px;list-style:none;">{_o}</ol>'
        '<details style="font-size:12.5px;">'
        '<summary style="cursor:pointer;color:#667eea;font-weight:600;">show answer</summary>'
        '<div style="margin-top:7px;padding:9px 11px;background:#f6f8ff;border-radius:9px;'
        f'color:#3c3f58;line-height:1.55;"><b>{chr(64 + _correct)}.</b> {_why}</div>'
        '</details></div>')
display(HTML(
    f'<div id="qz{_uid}" style="font-family:system-ui,Segoe UI,Roboto,sans-serif;'
    'border-radius:18px;border:1px solid #ecebff;'
    'background:linear-gradient(135deg,#f6f8ff,#fbf5ff);padding:19px;">'
    '<div style="font-size:15px;font-weight:700;color:#3c3f58;margin-bottom:5px;">'
    'Which stage broke?</div>'
    '<div style="font-size:12px;color:#6b6f80;margin-bottom:13px;">'
    'The same five options for each. Decide from the evidence above.</div>'
    f'{_blocks}</div>'))

### Why this is the whole problem

Three wrong answers. **Three unrelated causes, three different teams, three different fixes** —
and in two of the three the language model did nothing wrong at all.

Notice what let you tell them apart. Not the answer: the **trace**. A bug report saying "the
assistant gets things wrong" is unactionable. One saying "retrieval is not returning the Card
Charges Annex" is a ticket someone can close.

Doing that by hand, for three complaints, took a few minutes. The assistant answers **40,000
questions a month.** Everything from here is about turning that judgement into numbers — ones
you can track weekly, compare between suppliers, and put in front of a regulator.

In [ ]:
#@title 🧩 The five suspects, and who owns each (run me) { display-mode: "form" }
_uid = uuid.uuid4().hex[:8]
_suspects = [
    ("Never indexed", "The evidence was never loaded, or the copy that was loaded is stale.",
     "document owners", RED),
    ("Search missed it", "It is in the index. The search did not return it.",
     "retrieval vendor", RED),
    ("Ranked too low", "It was returned, then buried below the cut-off.",
     "reranking step", AMBER),
    ("Model ignored it", "It was in the shortlist and the answer went around it.",
     "model + prompt", AMBER),
    ("Model invented it", "The claim appears in no source at all.",
     "model + prompt", RED),
]
_c = "".join(f'''
  <div style="flex:1 1 185px;min-width:185px;background:#fff;border:1px solid #e6e8ee;
              border-left:4px solid {col};border-radius:12px;padding:13px 14px;">
    <div style="font-weight:650;font-size:13px;color:#22243a;margin-bottom:5px;">{i}. {t}</div>
    <div style="font-size:11.5px;color:#5b5f70;line-height:1.5;margin-bottom:7px;">{d}</div>
    <div style="font-size:10.5px;color:#8b8fa0;">owned by: <b>{w}</b></div>
  </div>''' for i, (t, d, w, col) in enumerate(_suspects, 1))
display(HTML(
    f'<div id="s{_uid}" style="font-family:system-ui,Segoe UI,Roboto,sans-serif;'
    'border-radius:18px;border:1px solid #ecebff;'
    'background:linear-gradient(135deg,#f6f8ff,#fbf5ff);padding:20px;">'
    '<div style="font-size:15px;font-weight:700;color:#3c3f58;">'
    'One symptom, five causes</div>'
    '<div style="font-size:12px;color:#6b6f80;margin:5px 0 14px;">'
    'A single quality score averages all five together, which is another way of saying it '
    'tells you nothing about any of them.</div>'
    f'<div style="display:flex;flex-wrap:wrap;gap:10px;">{_c}</div></div>'))

## The complaint we will follow

For each question the assistant searches its 721 passages, keeps the best handful, and hands
those to the language model to write a reply. *How* it picks them is the subject of Part 4 —
for now all that matters is that the passages below are the **real output of that search**,
not a selection of mine. They were computed once and frozen, so the notebook reproduces
exactly: ask the same question twice and the same passages come back.

One more, and this is the one the rest of the notebook measures. A customer asked what it
costs to send €2,000 to Poland; the assistant quoted the 1.7% conversion margin and answered
**€34.00**. The true cost is €53.90.

Here is its entire shortlist — all five passages, untrimmed, in rank order.

In [ ]:
q = BY_QID["q01"]
print(f"QUESTION:  {q['question']}\n")
print("CORRECT ANSWER:")
for line in textwrap.wrap(q["reference_answer"], 92):
    print(f"   {line}")
given = next(a for a in ANSWERS if a["answer_id"] == "q01_a2")
print("\nTHE ANSWER IT GAVE:")
for line in textwrap.wrap(given["text"], 92):
    print(f"   {line}")
print("\n" + "=" * 98)
print("WHAT THE ASSISTANT RETRIEVED — top 5, in rank order, complete text")
print("=" * 98)
for i, cid in enumerate(LIVE["q01"][:5], 1):
    p = BY_ID[cid]
    states = ", ".join(p["fact_ids"]) if p["fact_ids"] else "none"
    print(f"\n[{i}] {p['doc_title']}")
    print(f"    fees this passage states: {states}")
    print("    " + "-" * 90)
    for line in textwrap.wrap(p["text"], 90):
        print(f"    {line}")

In [ ]:
#@title 🎯 Check yourself — reading the Poland trace (run me) { display-mode: "form" }
_uid = uuid.uuid4().hex[:8]
QUIZ = [
    ("Of the three fees the correct answer needs, how many appear anywhere in those "
     "five passages?",
     ["None of them", "One", "Two", "All three"], 2,
     "Only the <b>1.7% conversion margin</b> — and it appears in four of the five passages. "
     "The fixed fee and the intermediary fee are never stated. The pages say where they can "
     "be found (&ldquo;the schedule of fees for your account&rdquo;, &ldquo;the correspondent "
     "banking annex&rdquo;), and the assistant did not follow those pointers."),
    ("How many distinct documents are those five passages drawn from?",
     ["Five", "Four", "Three", "One"], 3,
     "Three. The Poland page supplies passages 1, 2 and 3; Serbia and Romania one each. The "
     "shortlist is far less varied than five results suggests — and passage 3 states no fee "
     "at all, even though it comes from a document that does."),
]
_blocks = ""
for _n, (_q, _opts, _correct, _why) in enumerate(QUIZ, 1):
    _o = "".join(f'<li style="margin:3px 0;color:#3c3f58;">{chr(64 + _i)}. {_t}</li>'
                 for _i, _t in enumerate(_opts, 1))
    _blocks += (
        '<div style="background:#fff;border:1px solid #e6e8ee;border-radius:14px;'
        'padding:14px 16px;margin-bottom:11px;">'
        f'<div style="font-weight:650;font-size:13.5px;color:#22243a;margin-bottom:7px;">'
        f'{_n}. {_q}</div>'
        f'<ol style="margin:0 0 9px 18px;padding:0;font-size:12.5px;list-style:none;">{_o}</ol>'
        '<details style="font-size:12.5px;">'
        '<summary style="cursor:pointer;color:#667eea;font-weight:600;">show answer</summary>'
        '<div style="margin-top:7px;padding:9px 11px;background:#f6f8ff;border-radius:9px;'
        f'color:#3c3f58;line-height:1.55;"><b>{chr(64 + _correct)}.</b> {_why}</div>'
        '</details></div>')
display(HTML(
    f'<div id="qz{_uid}" style="font-family:system-ui,Segoe UI,Roboto,sans-serif;'
    'border-radius:18px;border:1px solid #ecebff;'
    'background:linear-gradient(135deg,#f6f8ff,#fbf5ff);padding:19px;">'
    '<div style="font-size:15px;font-weight:700;color:#3c3f58;margin-bottom:12px;">'
    f'Answer from the trace above, then reveal</div>{_blocks}</div>'))

### What the trace actually shows

The assistant was not hallucinating, and it was not ignoring its sources. Read passage 1 again:

> *"Three separate charges can apply to a payment outside the SEPA area. The fixed transfer fee
> is set out in **the schedule of fees for your account**. Where the payment is converted from
> euro, we apply a currency conversion margin of **1.7%** to the converted amount. Charges
> deducted by correspondent institutions are set out in **the correspondent banking annex**."*

The page states one of the three fees and **names exactly where the other two live.** Both of
those documents are in the index. Search returned neither.

So the answer was faithful to its sources and still wrong. That combination is the single most
important thing to understand about these systems, and it is the subject of Part 6.

In [ ]:
#@title Which of the three fees arrived, as the shortlist grows (run me) { display-mode: "form" }
rows = []
for fid, holders in q["fact_to_chunks"].items():
    rows.append({"fee the answer needs": fid,
                 "first stated in": textwrap.shorten(BY_ID[holders[0]]["doc_title"], 46,
                                                     placeholder="..."),
                 "passages stating it": len(holders)})
display(pd.DataFrame(rows))

print("If the assistant sent more passages to the model:\n")
for k in (3, 5, 10, 20, 50):
    got = [f.split(".")[-1] for f, cl in q["fact_to_chunks"].items()
           if set(cl) & set(LIVE["q01"][:k])]
    print(f"   top-{k:<3} {len(got)}/3 fees present   {got}")

Coverage is stuck at one fee of three until the twentieth result. The assistant sends five.

That is a diagnosis: **the evidence exists, search did not surface it, and no amount of work
on the language model would have fixed it.** Now let's turn that judgement into numbers — ones
you can track over time, compare between suppliers, and defend to a regulator.

---

# Part 2 — What counts as the right answer?

To grade the search you must first write down what it *should* have found. That sounds like
one decision. It is two, they are independent, and conflating them is how vendors end up
comparing numbers that were never comparable.

**Choice one — what unit do you mark as relevant?** The whole **document**, the individual
**passage**, or the **fee** the answer needs, wherever it lives.

**Choice two — how much of it must be found?** *Every* passage that states a needed fee, or
*any one* copy of it.

Look at the top three results again — all three come from the same Poland page:

| rank | passage | states a needed fee? |
|---|---|---|
| 1 | "How do I send money to Poland…" | yes — the 1.7% margin |
| 2 | "Worked example — a transfer of EUR 1,000…" | yes — the 1.7% margin again |
| 3 | "How long does a transfer to Poland take…" | **no** |

If you grade at document level, all three are hits — the Poland page is a relevant document.
Grade at passage level and the third is a miss. Same three results, different verdict, purely
from the unit you chose.

## Exercise 2. Grade the same result four different ways

**Task.** Write two functions and run them at both levels:

- `hit_at_k` — did **at least one** relevant item appear? The loosest standard there is, and
  the one a supplier means by "we retrieved the right document".
- `recall_at_k` — of **all** the items that should have been found, what fraction turned up?

In [ ]:
# 🎯 YOUR TURN — Exercise 2: grade the same result four different ways.
#
# 💭 Think first: both functions below get the SAME top-k list and the SAME answer key.
#    One asks "did anything relevant show up at all?", the other asks "how much of what
#    I wanted did I actually get?". On a question whose answer key holds 24 passages,
#    which of the two can still read a perfect 1.00 when the assistant found only one
#    of them? That gap is the whole point of this exercise.
#
# 🎯 Implement: replace the two `...` below. Everything under them already works.
#
#    Hints:
#      * `retrieved[:k]` is the top-k slice. Wrap both lists in `set(...)` so you can
#        use `&`, which gives you what the two have IN COMMON.
#      * An empty set is falsy, so `if set(a) & set(b):` reads as "they overlap".
#      * hit_at_k is a yes/no question -> return the float 1.0 or 0.0, nothing else.
#      * recall_at_k is a fraction -> how many gold items you found, divided by how many
#        gold items exist. `len()` of the intersection gives you the top half, and
#        `len(gold)` the bottom half.

def hit_at_k(retrieved, gold, k):
    """Did at least one relevant item make the top k? 1.0 or 0.0."""
    return ...


def recall_at_k(retrieved, gold, k):
    """Fraction of ALL the gold items that appear in the first k retrieved items."""
    if not gold:
        return float("nan")     # this question has no answer key -> nothing to score
    return ...


ranked_chunks = LIVE["q01"]
ranked_docs = list(dict.fromkeys(BY_ID[c]["doc_id"] for c in ranked_chunks))  # dedup, keep order

doc_hit = hit_at_k(ranked_docs, q["gold_doc_ids"], 3)
doc_recall = recall_at_k(ranked_docs, q["gold_doc_ids"], 3)
chunk_recall = recall_at_k(ranked_chunks, q["gold_chunk_ids"], 3)

print(f"a relevant DOCUMENT appeared        hit@3     {doc_hit:.2f}")
print(f"EVERY relevant document was found   recall@3  {doc_recall:.2f}"
      f"   ({len(q['gold_doc_ids'])} documents state a needed fee)")
print(f"EVERY relevant passage was found    recall@3  {chunk_recall:.2f}"
      f"   ({len(q['gold_chunk_ids'])} passages state a needed fee)")

Now the fourth yardstick. It is not a recall over passages at all — it asks whether **each
fee the answer needs** turned up *anywhere*, no matter which passage carried it.

In [ ]:
def fact_coverage_at_k(retrieved, fact_to_chunks, k):
    """Fraction of the fees the answer needs that have at least one passage in the top k."""
    if not fact_to_chunks:
        return float("nan")
    top = set(retrieved[:k])
    return float(np.mean([1.0 if set(v) & top else 0.0 for v in fact_to_chunks.values()]))


cov = fact_coverage_at_k(ranked_chunks, q["fact_to_chunks"], 3)
n_cov = sum(1 for v in q["fact_to_chunks"].values() if set(v) & set(ranked_chunks[:3]))

print("copies of each required fee in the corpus:")
for fid, holders in q["fact_to_chunks"].items():
    print(f"   {fid:<26} stated in {len(holders):>2} passage(s)")
print(f"   -> {len(q['gold_chunk_ids'])} passages count as relevant, "
      f"but only {len(q['fact_to_chunks'])} are needed to answer.\n")

summary = pd.DataFrame([
    {"what you asked for": "a relevant DOCUMENT appeared", "score": f"{doc_hit:.2f}",
     "why it lands there": "the loosest bar there is"},
    {"what you asked for": "EVERY relevant document", "score": f"{doc_recall:.2f}",
     "why it lands there": f"{len(q['gold_doc_ids'])} exist, 3 were needed"},
    {"what you asked for": "EVERY relevant passage", "score": f"{chunk_recall:.2f}",
     "why it lands there": f"{len(q['gold_chunk_ids'])} exist, 3 were needed"},
    {"what you asked for": "ANY ONE copy of each needed fee", "score": f"{cov:.2f}",
     "why it lands there": f"{n_cov} of {len(q['fact_to_chunks'])} fees arrived"},
])
display(summary)

### Reading the result

Four honestly-computed scores for one search, from **flawless** to **catastrophic**. They
differ for **two separate reasons**, and only one is about the unit.

**Reason one: the bar.** "A relevant document appeared" scores 1.00 because one did. Nothing
is wrong with the number; it answers a question nobody should have asked. The customer did not
need a document, they needed three numbers inside one.

**Reason two, the bigger one: redundancy.** The low scores are not evidence that passages are
too strict a unit. They are what happens when the bank states the conversion margin in **18**
passages, the fixed fee in **4**, and the intermediary fee in **2**. Twenty-four passages
therefore count as relevant, and an answer key demanding all twenty-four penalises the
assistant for failing to find copies of a fee it had already found.

Not a quirk of this question — across the test set each needed fee is stated **3.9 times** on
average:

In [ ]:
# Is q01 a one-off, or does the whole corpus repeat itself?
copies = [len(v) for x in SCORED for v in x["fact_to_chunks"].values()]
print(f"fees needed across the test set      : {len(copies)}")
print(f"average copies of each in the corpus : {np.mean(copies):.1f}"
      f"   (most duplicated: {max(copies)})")
print()
for k in (3, 5, 10):
    strict = np.nanmean([recall_at_k(LIVE[x["qid"]], x["gold_chunk_ids"], k) for x in SCORED])
    anyof = np.nanmean([fact_coverage_at_k(LIVE[x["qid"]], x["fact_to_chunks"], k)
                        for x in SCORED])
    print(f"  @{k:<3} every-copy recall {strict:.3f}   vs   any-one-copy coverage {anyof:.3f}"
          f"   gap {anyof - strict:+.3f}")

### What fact coverage actually is

Nothing exotic. **It is recall with any-of-N credit** — the same metric with the duplication
divided out. Each fee counts once, however many times the bank happens to have written it
down. That is why it tracked what happened to the customer while the other three did not.

**What this means for you.** A supplier quoting a retrieval score has made both choices on
your behalf and will usually tell you neither. Two questions, not one:

1. **What did you count as relevant** — documents, passages, or the fees the answer needs?
2. **Did you require every copy, or any one of them?**

Exhaustive labelling is the natural thing to produce when an answer key is built by scanning a
corpus for a fact — which is how most answer keys, including this one, get built. It quietly
turns a corpus that repeats itself into a verdict that your retrieval is bad.

---

# Part 3 — Did we find the evidence, and what does more of it cost?

The assistant sends the top **K** passages to the language model. K is a setting someone
picked. It is also a monthly invoice, and the two are the same decision.

Two numbers describe the trade:

- **Recall** — of the passages that should have been found, how many did we get? More is safer.
- **Precision** — of the passages we sent, how many were actually relevant? Higher means the
  model is not wading through noise.

## Exercise 3. Implement both, then price them

**Task.** Write `precision_at_k` and reuse your `recall_at_k`. Then the next cell sweeps K
across all 20 answerable questions and puts euros on the second axis.

In [ ]:
# 🎯 YOUR TURN — Exercise 3: implement precision, then look at what it costs.
#
# 💭 Think first: recall can only ever go UP as K grows — a bigger net catches more fish.
#    Precision almost always goes DOWN. Before you run the sweep, predict which of the
#    two the euro column below is going to track, and roughly where you would stop paying.
#
# 🎯 Implement: replace the single `...` in `precision_at_k`.
#
#    Hints:
#      * `relevance` is a dict mapping chunk_id -> grade, where the grade runs 0 to 3.
#        A passage counts as "relevant at all" when its grade is >= 1.
#      * A chunk that was never labelled is simply MISSING from that dict, which means
#        grade 0. Use `relevance.get(c, 0)` so a missing key returns 0 instead of raising.
#      * Count the relevant ones among the top k, then divide by k:
#        `sum(1 for c in retrieved[:k] if <the chunk is relevant>) / k`
#      * You do NOT need to rewrite recall_at_k — the one you wrote in Exercise 2 is
#        reused in the sweep below, along with fact_coverage_at_k, which is given.

def precision_at_k(retrieved, relevance, k):
    """Fraction of the first k retrieved passages that are relevant at all."""
    if k == 0:
        return float("nan")     # asking for the top 0 results is not a question
    return ...


Ks = [1, 2, 3, 5, 8, 10, 15, 20, 30, 50]
QUESTIONS_PER_MONTH, TOKENS_PER_PASSAGE, EUR_PER_MTOK = 40_000, 120, 0.60

sweep = pd.DataFrame([{
    "K": k,
    "precision": np.mean([precision_at_k(LIVE[x["qid"]], x["relevance"], k) for x in SCORED]),
    "recall": np.nanmean([recall_at_k(LIVE[x["qid"]], x["gold_chunk_ids"], k) for x in SCORED]),
    "fact coverage": np.nanmean([fact_coverage_at_k(LIVE[x["qid"]], x["fact_to_chunks"], k)
                                 for x in SCORED]),
    "EUR/month": k * TOKENS_PER_PASSAGE * QUESTIONS_PER_MONTH / 1e6 * EUR_PER_MTOK,
} for k in Ks])
display(sweep.round(3))

In [ ]:
#@title 📊 What each extra passage buys, and what it costs (run me) { display-mode: "form" }
fig, ax = plt.subplots(figsize=(7.4, 4.0))
ax.plot(sweep["K"], sweep["recall"], "o-", color=PURPLE, lw=2, label="recall (evidence found)")
ax.plot(sweep["K"], sweep["fact coverage"], "s-", color=GREEN, lw=2, label="fact coverage")
ax.plot(sweep["K"], sweep["precision"], "^-", color=AMBER, lw=2, label="precision (signal-to-noise)")
ax.set_xlabel("K — passages sent to the language model")
ax.set_ylabel("score")
ax.set_ylim(0, 1.02)
ax.set_xscale("log"); ax.set_xticks(Ks); ax.set_xticklabels(Ks)

cost = ax.twinx()
cost.plot(sweep["K"], sweep["EUR/month"], "--", color=RED, lw=1.6, label="cost")
cost.set_ylabel("€ per month at 40,000 questions", color=RED)
cost.tick_params(axis="y", colors=RED)
cost.spines["right"].set_visible(True); cost.spines["right"].set_color(RED)
cost.spines["top"].set_visible(False)

h1, l1 = ax.get_legend_handles_labels(); h2, l2 = cost.get_legend_handles_labels()
ax.legend(h1 + h2, l1 + l2, loc="lower right", fontsize=8, framealpha=0.95)
ax.set_title("More evidence is always better, and never free", fontsize=11)
plt.tight_layout(); plt.show()

In [ ]:
multi = [x for x in QUESTIONS if x["type"] == "multi_fact"]
print("fact coverage on the questions that need several fees (like the Poland one):\n")
for k in (3, 5, 10, 20, 50):
    c = np.nanmean([fact_coverage_at_k(LIVE[x["qid"]], x["fact_to_chunks"], k) for x in multi])
    bar = "█" * int(round(c * 34))
    print(f"   K={k:<3} {c:5.3f}  {bar}")

### Reading the result

The assistant currently runs at **K=5**. Look at what that costs you:

| | K=5 (today) | K=20 | K=50 |
|---|---|---|---|
| fact coverage, all questions | 0.767 | **0.967** | 0.983 |
| fact coverage, multi-fee questions | **0.417** | **0.917** | 0.958 |
| cost per month | €14 | €58 | €144 |

On the questions that need several fees — the Poland question and its kind — **today's setting
finds all the required fees 42% of the time.** Going to K=20 takes that to 92% for about €43
more a month. Going on to K=50 adds another €86 a month to buy 4 more points.

That is not a technical tuning question. It is: *the fix for the thing that cost us 200
refunds is a €43-a-month line item.* Precision falls as K rises, which is the real reason not
to set K=50 and forget it — but at these numbers, K=5 is indefensible.

**Note what Hit@K would have told you.** "Did we find *at least one* relevant passage?" is
1.00 at K=5. It cannot tell "found one of three fees" from "found all three" — which is
precisely the failure that put you here. Retire it for this use case.

---

# Part 4 — Did we find it *first*?

Finding the right passage at position 9 is nearly as bad as not finding it. The model reads
the top of the list most carefully, and you pay to send it the rest.

So we need a score that rewards good things being **early**, not merely present. The standard
one discounts each position by how far down it sits:

$$DCG@K = \sum_{i=1}^{K} \frac{rel_i}{\log_2(i+1)}$$

where $rel_i$ is how useful the passage at position $i$ is (we grade 3 = essential, 2 = useful,
1 = marginal, 0 = irrelevant), and the $\log_2(i+1)$ underneath shrinks the contribution of
anything further down the page. In practice the weights are:

| position | 1 | 2 | 3 | 4 | 5 |
|---|---|---|---|---|---|
| what it's worth | 1.00 | 0.63 | 0.50 | 0.43 | 0.39 |

An essential passage at position 1 is worth **two and a half times** the same passage at
position 5.

Raw DCG is not comparable across questions — a question with six relevant passages can score
higher than one with two, without being better served. So we divide by the best score that
ordering *could* have achieved (sort your own labels perfectly): that ratio is **nDCG**,
and it runs 0 to 1.

## Exercise 4. Implement DCG and nDCG

In [ ]:
# 🎯 YOUR TURN — Exercise 4: implement DCG, then nDCG.
#
# 💭 Think first: the discount weights for ranks 1-5 are 1.00, 0.63, 0.50, 0.43, 0.39.
#    A passage graded 3 sitting at rank 5 contributes 3 x 0.39 = 1.17; move it to rank 1
#    and it contributes 3.00. So DCG rewards putting the good stuff first. But a raw DCG
#    of 4.2 means nothing on its own — 4.2 out of WHAT? Hold that question; it is exactly
#    what the second function fixes.
#
# 🎯 Implement: replace the three `...` below.
#
#    Hints for dcg_at_k — `relevances` is a list of grades already IN RANK ORDER,
#    e.g. [3, 3, 2, 0, 0] means "rank 1 was graded 3, rank 2 was graded 3, ...":
#      * The discount for position i (counting from 0) is 1 / log2(i + 2). Check it:
#        i=0 gives 1/log2(2) = 1.00, i=1 gives 1/log2(3) = 0.63. Those are the weights
#        above. Use `np.log2`.
#      * So each item contributes `rel / np.log2(i + 2)`, and you want the SUM of those.
#      * `enumerate(relevances)` hands you (i, rel) pairs, which is exactly what you need:
#        `sum(... for i, rel in enumerate(relevances))`
#
#    Hints for ndcg_at_k — this is one division, once you build the two halves:
#      * `actual`: the grades of what we really returned, in the order we returned them.
#        Look up each retrieved chunk's grade -> `[relevance.get(c, 0) for c in
#        retrieved[:k]]` -> then pass that list to dcg_at_k().
#      * `ideal` : the SAME labels, sorted best-first — the score a perfect ranker would
#        have got -> `sorted(relevance.values(), reverse=True)[:k]` -> pass to dcg_at_k().
#      * nDCG is then just actual / ideal, which lands between 0 and 1. The guard against
#        dividing by zero is already written for you on the last line.

def dcg_at_k(relevances):
    """Sum of usefulness, each discounted by how far down the list it sits."""
    return ...


def ndcg_at_k(retrieved, relevance, k):
    """DCG of what we returned, divided by the DCG of the best possible ordering."""
    actual = ...
    ideal = ...
    return actual / ideal if ideal else float("nan")


# A worked example: the same five passages, two different orders.
good = [3, 3, 2, 0, 0]
bad = [0, 0, 2, 3, 3]
print(f"  essential results first : DCG = {dcg_at_k(good):.3f}")
print(f"  essential results last  : DCG = {dcg_at_k(bad):.3f}")
print(f"  same passages, {dcg_at_k(good) / dcg_at_k(bad):.1f}x the score, purely from the ordering")

### Where a reranker comes in

The search is fast because it never actually reads the question and the passage *together*. It
turned every passage into 384 numbers months ago, turns your question into 384 numbers now, and
compares the two lists. One multiplication across the whole corpus — microseconds, and
somewhat blunt, because each passage was summarised without any idea of what would be asked.

A **reranker** is a different kind of model. It takes the question and one passage **glued
into a single piece of text** and runs a full language model over the pair to produce one
relevance score. The passage has no fixed representation at all here: how it is read depends
on the question being asked. Much sharper — and impossible to prepare in advance, for exactly
that reason.

| | the search | the reranker |
|---|---|---|
| reads question and passage | separately | **together, as one input** |
| can be computed in advance | yes, once, offline | **no — it needs the question** |
| cost for this corpus | ~0.3 ms for all 721 | **~0.9 s for 50 pairs** |

That cost difference is the entire reason for the two-stage design. Running the reranker over
all 721 passages would take about 13 seconds per question; over a real bank's millions of
passages it is simply impossible. So the cheap model narrows 721 down to 50, and the expensive
one only ever looks at those 50.

```
721 passages  --( search, ~0.3 ms )-->  50 candidates  --( reranker, ~0.9 s )-->  5 to the model
```

The model doing the reranking here is `cross-encoder/ms-marco-MiniLM-L-6-v2`, a small
six-layer model trained to score exactly this kind of question-and-passage pair. Its output is
a raw score, unbounded — on the Poland question the 50 candidates span −11.3 to +3.0. Only the
*ordering* carries meaning; the numbers are not probabilities and cannot be compared against
the similarity scores from the search step.

> **A note on how this notebook runs.** The reranker was run **once, when the dataset was
> built**, and every ranking it produced was frozen into `runs.json`. The cells below read
> those frozen rankings rather than running the model, which is why nothing here downloads a
> model or takes a minute to execute. The code that produced them is in
> `03_cx_generate_bank_corpus.py`; retrieval is deterministic, so re-running it reproduces
> these rankings exactly.

Here is what it does to the live pipeline:

In [ ]:
def evaluate(ranking_by_q, k_recall=50, k_ndcg=5):
    return {
        "Recall@50": np.nanmean([recall_at_k(ranking_by_q[x["qid"]], x["gold_chunk_ids"], k_recall)
                                 for x in SCORED]),
        "nDCG@5": np.nanmean([ndcg_at_k(ranking_by_q[x["qid"]], x["relevance"], k_ndcg)
                              for x in SCORED]),
    }

before = evaluate(RUNS["retriever_dense"])
after = evaluate(RUNS["pipeline_dense_reranked"])
display(pd.DataFrame([{"pipeline": "search only", **before},
                      {"pipeline": "search + reranker", **after}]).round(3))

### Reading the result

**Recall did not move. Not by a rounding error — not at all.** Ordering improved by about
14%.

That is not a disappointing result, it is the defining property of a reranker: **it can only
reorder what it was handed. It can never add evidence.** If a fee was not in the 50 candidates,
no amount of reranking conjures it up.

Which gives you a clean division of responsibility to hold two vendors to:

- the **search** is responsible for *recall* — is the evidence in the pile at all?
- the **reranker** is responsible for *ordering* — is the best of it at the top?

A supplier who reports one blended quality score for both has hidden which of the two is
broken. Ask for them separately.

## Exercise 5. Choose what you would actually ship

You now have three curves: recall and coverage against K, cost against K, and the ordering
gain from reranking.

**Task.** Pick the operating point you would sign off on — how many candidates the reranker
should consider, and how many passages get sent to the model — and justify it in two
sentences. There is no single right answer here; there is a defensible one and an
indefensible one.

In [ ]:
# 🎯 YOUR TURN — Exercise 5: choose the operating point you would actually ship.
#
# 💭 Think first: this is the one exercise in the notebook with no single right answer,
#    and that is deliberate — it is a judgement about money and risk, not a formula.
#    There IS a defended answer in the solutions notebook, but yours can differ from it
#    and still be completely right, as long as you can defend it the same way.
#
# 🎯 Implement: replace the four `...` below with your own numbers and your own reasoning.
#    Nothing here is graded by a computer, so you cannot get it "wrong" — you can only
#    fail to justify it.
#
#    Hints — scroll back to two things before you answer:
#      * the K sweep table in Part 3 (the `sweep` DataFrame): what happens to fact
#        coverage between K=5 and K=20, and what does the EUR/month column say that
#        change costs?
#      * the reranker table in Part 4: recall@50 is already 0.944, and the reranker
#        cannot add evidence that is not in the shortlist.
#
#    A defensible answer usually cites three things:
#      1. a number from the sweep  (what does more K actually buy you?)
#      2. a number from the cost column  (what does that buy cost per month?)
#      3. what it costs to be wrong  (what does ONE refunded complaint cost the bank?)
#
#    For the third entry, read the euro figure for your chosen K straight off the sweep.

choice = {
    "candidates for the reranker": ...,
    "passages sent to the model": ...,
    "cost at 40,000 questions/month": ...,
}
why = ...          # two sentences: what you chose, and the numbers that justify it


for k, v in choice.items():
    print(f"  {k:<34} {v}")
print(textwrap.fill("\nWhy: " + str(why), 92))

---

# Part 5 — The best parts do not make the best machine

**Vendor A is what you run today.** Its search is the one behind every trace you have looked
at so far, and its headline number — recall@50, "did the evidence make it into the shortlist
at all" — is the figure in your board pack.

A challenger, **Vendor B**, asks for a trial. You run their search over your own documents,
measure it the way you measure your own, and get this.

In [ ]:
component = pd.DataFrame([
    {"search component": "Vendor A — what you run today",
     "Recall@50": np.nanmean([recall_at_k(RUNS["retriever_dense"][x["qid"]],
                                          x["gold_chunk_ids"], 50) for x in SCORED])},
    {"search component": "Vendor B — the challenger",
     "Recall@50": np.nanmean([recall_at_k(RUNS["retriever_diverse"][x["qid"]],
                                          x["gold_chunk_ids"], 50) for x in SCORED])},
]).round(3)
display(component)
print("The challenger is WORSE on the number you track — 0.77 against your 0.94.")

**Commit before you scroll.** The challenger finds *less* of the evidence than your
current supplier, on your own data, measured your own way. Do you end the trial there?

That is the decision most people make, and it takes about four seconds. Before you make it,
one more measurement — the same two searches, each with your reranker bolted on, scored on
what the pipeline is actually *for*: did the fees the answer needs reach the model?

In [ ]:
def endtoend(run_key):
    return {
        "fact coverage@5": np.nanmean([fact_coverage_at_k(RUNS[run_key][x["qid"]],
                                                          x["fact_to_chunks"], 5) for x in SCORED]),
        "multi-fee questions": np.nanmean([fact_coverage_at_k(RUNS[run_key][x["qid"]],
                                                              x["fact_to_chunks"], 5)
                                           for x in multi]),
    }

full = pd.DataFrame([
    {"full pipeline": "A (today) + reranker",
     "Recall@50": component.loc[0, "Recall@50"], **endtoend("pipeline_dense_reranked")},
    {"full pipeline": "B (challenger) + reranker",
     "Recall@50": component.loc[1, "Recall@50"], **endtoend("pipeline_diverse_reranked")},
]).round(3)
display(full)

In [ ]:
#@title 📊 Component winner vs pipeline winner (run me) { display-mode: "form" }
fig, axes = plt.subplots(1, 2, figsize=(8.2, 3.4))
labels = ["A (today)", "B (challenger)"]
for ax, (col, title) in zip(axes, [("Recall@50", "The number in your board pack\n(evidence in the shortlist)"),
                                   ("multi-fee questions",
                                    "What the pipeline is for\n(all needed fees reach the model)")]):
    vals = list(full[col])
    win = int(np.argmax(vals))
    bars = ax.bar(labels, vals, color=[GREEN if i == win else "#c9ccd6" for i in range(2)], width=0.55)
    ax.bar_label(bars, fmt="%.3f", fontsize=9, padding=2)
    ax.set_ylim(0, max(vals) * 1.28); ax.set_title(title, fontsize=10); ax.set_ylabel(col)
plt.tight_layout(); plt.show()

### What the two vendors actually do differently

Before explaining the result, it is worth knowing what separates them — because it is smaller
than "two suppliers" suggests. **Both use the same embedding model and get the same similarity
scores.** What differs is only *which fifty they keep*:

- **Vendor A** sorts all 721 passages by similarity and takes the top fifty. That is the whole
  policy.
- **Vendor B** picks them one at a time, and each time it picks, it subtracts a penalty for how
  similar a candidate is to something it has *already chosen*:

  ```
  score = 0.55 × (similarity to the question)  −  0.45 × (similarity to the closest passage already picked)
  ```

  So a passage that is highly relevant but nearly identical to one already in the list gets
  pushed down. Vendor B deliberately declines to fill its shortlist with copies.

That is the entire difference. Same model, same scores, different selection rule. Run both
below and look at what comes out.

In [ ]:
# Both policies, run live on the stored vectors. No model download: the passage vectors and
# the 25 question vectors were computed once and ship with the data.
E = np.load(f"{DATA_DIR}/embeddings.npy")          # 721 passages x 384
QE = np.load(f"{DATA_DIR}/query_embeddings.npy")   # 25 questions x 384
CHUNK_IDS = [c["chunk_id"] for c in chunks]
QIDX = {x["qid"]: i for i, x in enumerate(QUESTIONS)}


def vendor_a(sim, k=50):
    """Plain top-k by similarity."""
    return list(np.argsort(-sim)[:k])


def vendor_b(sim, k=50, lam=0.55):
    """Same scores, but penalise a candidate for resembling what is already picked."""
    picked, pool = [], list(np.argsort(-sim)[:400])
    while len(picked) < k and pool:
        if not picked:
            best = pool[0]
        else:
            already = E[picked]
            scored = [lam * sim[i] - (1 - lam) * float(np.max(E[i] @ already.T)) for i in pool]
            best = pool[int(np.argmax(scored))]
        picked.append(best)
        pool.remove(best)
    return picked


sim = E @ QE[QIDX["q01"]]                       # similarity of every passage to the question
A = [CHUNK_IDS[i] for i in vendor_a(sim)]
B = [CHUNK_IDS[i] for i in vendor_b(sim)]
print(f"reproduces the frozen rankings exactly:  "
      f"A {A == RUNS['retriever_dense']['q01']},  B {B == RUNS['retriever_diverse']['q01']}")

In [ ]:
#@title 📊 The shape of the two shortlists (run me) { display-mode: "form" }
from collections import Counter

fig, axes = plt.subplots(1, 2, figsize=(9.6, 3.6), sharey=True)
for ax, (name, lst, colour) in zip(axes, [("Vendor A — top 50 by similarity", A, PURPLE),
                                          ("Vendor B — same scores, no near-duplicates", B, GREEN)]):
    cnt = Counter(BY_ID[c]["doc_title"] for c in lst)
    top = cnt.most_common(8)
    ax.barh([textwrap.shorten(t, 34, placeholder="…") for t, _ in top][::-1],
            [n for _, n in top][::-1], color=colour)
    ax.set_title(f"{name}\n{len(cnt)} distinct documents among its 50", fontsize=9.5)
    ax.set_xlabel("passages contributed")
    ax.tick_params(axis="y", labelsize=7.5)
plt.tight_layout(); plt.show()

for name, lst in [("Vendor A", A), ("Vendor B", B)]:
    cnt = Counter(BY_ID[c]["doc_id"] for c in lst)
    corridor = sum(v for k, v in cnt.items() if k.startswith("faq_sending_money"))
    print(f"{name}: {len(cnt)} distinct documents | {corridor}/50 are 'sending money to X' "
          f"pages | most-repeated document supplies {max(cnt.values())} passages")

In [ ]:
# The question that matters: did each of the three needed fees make it into the shortlist?
rows = []
for fid, holders in BY_QID["q01"]["fact_to_chunks"].items():
    pa = sorted(A.index(c) + 1 for c in holders if c in A)
    pb = sorted(B.index(c) + 1 for c in holders if c in B)
    rows.append({"fee the answer needs": fid.split(".")[-1],
                 "Vendor A — positions": pa[:3] if pa else "NOT IN ITS 50",
                 "Vendor B — positions": pb[:3] if pb else "NOT IN ITS 50"})
display(pd.DataFrame(rows))

### Reading the result

**The supplier you would have rejected is the better system.** On exactly the questions that
caused this review, your current search delivers all the required fees 42% of the time; the
challenger you were about to dismiss manages 52%.

Look at the last table for the reason, because it is sharper than "less variety". Vendor A's
fifty slots are consumed by the corridor pages — one document alone supplies thirteen of them —
and **the intermediary bank fee never enters its shortlist at all.** Not ranked low: absent.
No reranker can recover it, because reranking only reorders what it was handed. Vendor B,
declining the duplicates, has room for it at position 21, from where the reranker can lift it.

So the duplicates did not merely *dilute* Vendor A's shortlist. They **crowded out a distinct
fee**.

And now the sting: Vendor A still scores **higher recall** (0.944 against 0.770). Recall counts
gold *passages*, and eighteen of the twenty-four for this question are copies of the same
conversion margin. Vendor A collects more of those copies, and is rewarded for it — while
dropping an entire fee that Vendor B finds.

**Component metrics are for diagnosis. End-to-end metrics are for selection.**

Two things follow, and both are things you can say in a supplier meeting:

1. A component number — theirs *or yours* — is not evidence about the assembled system.
   Components interact, and here your own headline metric was the thing pointing you at the
   wrong answer. Test the whole pipeline on your own data before you trust either number.
2. Be precise about what "end-to-end" means. Had we compared these two on ordering quality
   instead of on whether the needed facts arrived, Vendor A would have looked fine. **Pick the
   measure that matches the job the system does** — here, "did the answer get all three fees",
   because that is what the customer was owed.

---

# Part 6 — Correct is not the same as supported

Everything so far has been about finding evidence. This part is about what the model does with
it, and it turns on a distinction that costs banks money:

- **Correct** — does the answer match reality?
- **Supported** (or *grounded*) — does the answer only say things its sources actually say?

These come apart in both directions, and each combination is a different problem with a
different owner.

In [ ]:
#@title 📊 The four kinds of answer (run me) { display-mode: "form" }
from collections import Counter
qd = Counter(a["quadrant"] for a in ANSWERS)
_uid = uuid.uuid4().hex[:8]
_cells = [
    ("correct_grounded", "Correct <b>and</b> supported", "What you want.", GREEN),
    ("correct_ungrounded", "Correct, <b>not</b> supported",
     "Right answer, invented extras — or right by luck.", AMBER),
    ("wrong_grounded", "Wrong, <b>but</b> supported",
     "Quoted its sources faithfully. The sources were wrong or incomplete.", AMBER),
    ("wrong_ungrounded", "Wrong <b>and</b> unsupported", "Made it up.", RED),
]
_h = "".join(f'''
  <div style="flex:1 1 210px;min-width:210px;background:#fff;border:1px solid #e6e8ee;
              border-left:4px solid {col};border-radius:12px;padding:13px 15px;">
    <div style="font-size:13px;color:#22243a;margin-bottom:4px;">{lab}</div>
    <div style="font-size:26px;font-weight:700;color:{col};line-height:1.1;">{qd.get(key,0)}</div>
    <div style="font-size:11.5px;color:#5b5f70;line-height:1.5;margin-top:4px;">{d}</div>
  </div>''' for key, lab, d, col in _cells)
display(HTML(f'''
<div id="q{_uid}" style="font-family:system-ui,Segoe UI,Roboto,sans-serif;border-radius:18px;
     border:1px solid #ecebff;background:linear-gradient(135deg,#f6f8ff,#fbf5ff);padding:20px;">
  <div style="font-size:15px;font-weight:700;color:#3c3f58;">
     {len(ANSWERS)} answers from the pilot, labelled by hand</div>
  <div style="font-size:12px;color:#6b6f80;margin:5px 0 14px;">
     The two amber boxes are the ones nobody plans for.</div>
  <div style="display:flex;flex-wrap:wrap;gap:11px;">{_h}</div>
</div>'''))

### The row that explains the whole incident

Take the answer that started this — the €34.00 one — and look at it from both directions
at once.

In [ ]:
a = next(x for x in ANSWERS if x["answer_id"] == "q01_a2")
print("THE ANSWER THE CUSTOMER GOT")
for line in textwrap.wrap(a["text"], 92):
    print(f"   {line}")

print(f"\nTHE {len(a['context_chunk_ids'])} PASSAGES IT WAS SHOWN")
for cid in a["context_chunk_ids"]:
    print(f"   [{cid}] {textwrap.shorten(BY_ID[cid]['text'], 140, placeholder=' …')}")

print("\nTHE TWO VERDICTS, EACH A PLAIN YES OR NO")
print(f"   supported by those passages?  {'YES' if a['human']['grounded'] else 'no':<4}"
      f"  — it only said things the text it was given says")
print(f"   correct?                      {'yes' if a['human']['correct'] else 'NO':<4}"
      f"  — the customer was quoted €34.00 and owed €53.90")

**Every single thing that answer said was true and came from its source.** It was still
wrong, because the source it was handed held one of the three fees.

A faithfulness check passes this answer, and it is *right to*. This is the single most
important idea in the notebook: a green light on one instrument means "my stage is fine",
never "the system is fine."

Notice how little machinery that took. Both verdicts are a plain yes or no, recorded once per
answer by a person who read the answer against the passages it was shown — the same shape of
judgement a compliance reviewer already makes. Forty-two answers, two questions each, one
afternoon.

That is what produced the four boxes above, and here is one real answer from each of them.

In [ ]:
rows = []
for key, label in [("correct_grounded", "correct + supported"),
                   ("wrong_grounded", "wrong + supported"),
                   ("correct_ungrounded", "correct + unsupported"),
                   ("wrong_ungrounded", "wrong + unsupported")]:
    ex = next(x for x in ANSWERS if x["quadrant"] == key)
    rows.append({"kind": label,
                 "example": textwrap.shorten(ex["text"], 74, placeholder="…"),
                 "supported": "yes" if ex["human"]["grounded"] else "no",
                 "correct": "yes" if ex["human"]["correct"] else "no"})
display(pd.DataFrame(rows))

Notice the second row: **supported "yes", correct "no"**. No amount of grounding checking
will ever flag it, because grounding is not what is broken.

### So what *would* have caught it?

The wrong answer came from a fee schedule that was replaced in January and never removed from
the index. That is not a search problem or a model problem — it is a problem with what got
loaded. Two cheap checks on the index itself would have found it before a customer did.

In [ ]:
# Check 1 — is every fact the test set needs present in the index at all?
missing = []
for qq in QUESTIONS:
    for fid, holders in qq.get("fact_to_chunks", {}).items():
        if not holders:
            missing.append((qq["qid"], fid))
print(f"CHECK 1  facts required by a test question but absent from the index: {len(missing)}")

# Check 2 — does the index hold two passages stating the same fee with different dates?
conflicts = []
for i, x in enumerate(chunks):
    if not x["fact_ids"] or not x["effective_date"]:
        continue
    for y in chunks[i + 1:]:
        if (x["fact_ids"] == y["fact_ids"] and y["effective_date"]
                and x["effective_date"] != y["effective_date"]):
            conflicts.append((x, y))
print(f"CHECK 2  passage pairs stating the same fees with different effective dates: "
      f"{len(conflicts)}\n")

x, y = conflicts[0]
for p in (x, y):
    print(f"   [{p['effective_date']}] {p['doc_title']}")
    print(f"       {textwrap.shorten(p['text'], 104, placeholder=' …')}")

### Reading the result

Thirty-four pairs of passages in the index state the same fee with two different effective
dates. Every one of them is a wrong answer waiting to be quoted, faithfully, to a customer.

The check is four lines. It needs no model, no test questions, and no labelled data — just the
index. It would have caught the incident in Part 1's second trace **before launch**.

### Which leaves one problem

Every number in this part rests on those two hand-written verdicts. Someone read forty-two
answers against their sources and wrote down *supported* and *correct*. That is the most
reliable evaluation in the notebook, and it is the only one that does not survive contact with
a release schedule: the moment you change the model, the prompt, the chunk size or the index,
every one of those verdicts is about a system that no longer exists, and somebody has to read
forty-two answers again.

Part 7 is about handing that reading to a model — and about how you would know whether the
model reads them the way you do.

---

# Part 7 — Using a model as the evaluator

Part 6 ended with two hand-written verdicts per answer. This part replaces the hand with a
model, and then checks whether that was safe.

## First, be clear about what actually gets graded

A common confusion is worth killing immediately. **You are not grading all 40,000 answers the
assistant produces each month.** You are grading a *test set*: the 25 questions and 42 answers
in this notebook, each with a reference answer written in advance. It is the same idea as a
regression test suite — a fixed, known set you re-run after every change to see what moved.

So how much work is that really? The honest arithmetic, at the rate the labelling in Part 6
actually took (about six minutes an answer):

| what you grade | answers | human time |
|---|---|---|
| the test set, once | 42 | ~4 hours — **an afternoon** |
| the test set, re-run every release | 42 × every change | an afternoon **per release** |
| 1% of live traffic | ~400 / month | ~40 hours — **a week of someone's month** |
| every answer the assistant gives | 40,000 / month | ~4,000 hours — **24 full-time people** |

Read the first row before the last one. **A human absolutely can grade this test set** — one
did, and that is where the labels in Part 6 came from. The case for automating is not that the
job is impossible. It is the second row: you have to redo it *every time anything changes*, and
an afternoon of turnaround between "we changed the chunk size" and "we know if that helped" is
what stops teams from measuring at all. Automate it and the same suite runs in a minute, so it
runs on every change rather than on the changes someone found time for.

The bottom two rows are the second reason: a test set only contains questions you thought to
ask. Sampling real traffic catches what it never occurred to you to test — and *that* is where
hand-grading genuinely stops scaling.

## Second, the obvious objection

A model grading a model sounds circular. It is worth being precise about why it is not:

- **Verification is a far easier task than generation.** Producing the answer meant finding
  three fees among 721 passages. Grading it means reading one short answer against a handful
  of passages that are handed to you.
- **The grader is given things the writer never had** — the retrieved passages *and*, for
  correctness, the reference answer.

That is why it *can* work. Whether it *does* is a question you settle with measurements, not
with reasoning, and that is the rest of this part.

## What the grader was actually asked

These are the two prompts that produced every score below, reproduced as they were sent. Two
separate calls per answer, never one — merging them collapses the distinction Part 6 spent its
length establishing into a single unreadable number.

Both calls share one system instruction: *"You are a strict evaluator. Reply with JSON only.
No prose, no code fences."*

**Call 1 — supported?** Gets the passages the answer was shown. **Does not get the reference
answer**, because a grader that knows the right answer stops checking groundedness and starts
checking correctness.

```
Decide whether the ANSWER is supported by the SOURCE PASSAGES below.
Judge ONLY against these passages - do not use outside knowledge.

SOURCE PASSAGES:   [the passages that answer was actually shown]
QUESTION:          [the customer's question]
ANSWER:            [the answer being graded]

Judge only what the answer actually states - not what it implies, and not
what it leaves out.

Something is supported if the passages state it, or if it follows by
arithmetic from figures the passages state. Example: a passage gives a fee
of 2% and the answer says a EUR 500 payment carries EUR 10.00 - that is
supported, the figure is computed from the passages, not invented.

An answer is grounded if everything it states is supported. An answer can be
grounded and still incomplete - do NOT mark it ungrounded for omitting
something the question needed.

JSON: {"grounded": "yes" or "no", "score": 1-5, "rationale": "one sentence"}
```

**Call 2 — correct?** Gets the reference answer. **Does not get the passages**, because whether
an answer is right does not depend on what the search happened to retrieve.

```
Grade the ANSWER against the REFERENCE ANSWER.

QUESTION:          [the customer's question]
REFERENCE ANSWER:  [what the answer should have been]
ANSWER TO GRADE:   [the answer being graded]

JSON: {"correct": "yes" or "no", "score": 1-5, "rationale": "one sentence"}
```

Three things in that first prompt are worth your attention, because each one is a **definition
you are choosing**, not a technical detail:

- *"not what it implies, and not what it leaves out"* — without this the grader marks
  incomplete answers ungrounded, and you can no longer tell the two failures apart. It is the
  cold open's distinction, written into the instructions.
- *the arithmetic sentence* — the assistant's job is adding up fees, so a grader that cannot
  accept a computed total would fail every correct answer it sees. (The figures in the example
  appear in no question in the set, so it teaches the rule without handing over an answer.)
- *"grounded and still incomplete"* — said twice, in two ways, because it is the rule the
  grader breaks most often.

**Nobody hands this grader a list of claims.** It gets the answer as prose and has to work out
for itself what the answer asserts — the same thing the person in Part 6 did. That matters
when you read the agreement number: the grader is doing the whole job, not the second half of
it.

And note what it never sees: **which passages were *supposed* to be retrieved.** That is the
answer key from Parts 2–5, and it belongs to a different question. This grader judges answers,
nothing else.

In [ ]:
# The grader run reported here is the one that received no claim list — it read each
# answer as prose and decided for itself what it asserted.
GRADER = next(m for m in META["judge_models"] if "v2" in m)
MODEL_NAME = GRADER.split(" (")[0]


def verdict(x):
    """Turn the grader's "yes"/"no" into a 1/0 that lines up with the human labels."""
    return 1 if str(x).strip().lower() == "yes" else 0


print(f"grader model  : {MODEL_NAME}")
print(f"answers scored: "
      f"{sum(1 for a in ANSWERS if (a.get('judges') or {}).get(GRADER))}/{len(ANSWERS)}"
      f"   — produced once, in advance, and frozen into the data.")

ex = next(a for a in ANSWERS if a["answer_id"] == "q01_a2")
j = ex["judges"][GRADER]
print("\nWhat it returned for the answer that started this notebook:\n")
print(f"  supported?  {j['faithfulness']['grounded'].upper():<4}  {j['faithfulness']['rationale']}")
print(f"  correct?    {j['correctness']['correct'].upper():<4}  {j['correctness']['rationale']}")

**Supported yes, correct no** — and read its two one-line reasons. It says the answer
computes the 1.7% margin correctly from the passage it was given, and that the answer leaves
out the fixed fee and the intermediary fee that the reference includes.

That is the whole of Part 6, drawn correctly, by a small free model, from prose, for a fraction
of a cent. No claim list, no answer key, no human in the loop.

One case is an anecdote. Here is every answer.

In [ ]:
#@title Every answer: what the human said, what the grader said (run me) { display-mode: "form" }
rows = []
for a in ANSWERS:
    j = (a.get("judges") or {}).get(GRADER) or {}
    c, f = (j.get("correctness") or {}), (j.get("faithfulness") or {})
    gc, gf = verdict(c.get("correct")), verdict(f.get("grounded"))
    rows.append({
        "answer": a["answer_id"],
        "the answer it gave": textwrap.shorten(a["text"], 54, placeholder="…"),
        "human: correct?": "yes" if a["human"]["correct"] else "no",
        "grader: correct?": "yes" if gc else "no",
        "human: supported?": "yes" if a["human"]["grounded"] else "no",
        "grader: supported?": "yes" if gf else "no",
        "": "" if (gc == a["human"]["correct"]
                   and gf == a["human"]["grounded"]) else "← DIFFER",
    })
display(pd.DataFrame(rows))

## Exercise 6. Measure the grader

Reading 42 rows is not a method. Before you let this thing gate a release, you need one number:
**how often does it reach the same verdict a person did?**

Both sides are now a plain yes or no, so this is just counting — but count the two ways of
disagreeing *separately*, because they are not equally bad:

- a **false accept** is the grader passing something a human failed. That one reaches a
  customer.
- a **false reject** is the grader failing something a human passed. That one costs somebody
  an hour of review.

A single "agreement: 90%" hides which of those you are buying.

**Task.** Write `agreement`, taking the grader's yes/no verdicts and the human's, and returning
the fraction that match plus those two counts kept apart.

In [ ]:
# 🎯 YOUR TURN — Exercise 6: measure the grader.
#
# 💭 Think first: suppose this grader agreed with the human on 90% of answers. Is that
#    good enough to gate a release? It depends entirely on WHICH 10% it got wrong. One
#    kind of mistake sends a wrong bank fee to a customer; the other wastes an analyst's
#    afternoon. Decide which one you would rather your gate made BEFORE you compute it —
#    that is why the function returns them separately instead of one accuracy number.
#
# 🎯 Implement: replace the three `...` below.
#
#    Hints — `grader` and `human` are two equally long lists of 1s and 0s (1 = yes,
#    0 = no), lined up by position, so `grader[i]` and `human[i]` are two verdicts on
#    the SAME answer:
#      * `zip(grader, human)` walks both lists in step and hands you (g, h) pairs. Every
#        line below is some version of "count the pairs where <something> is true".
#      * agreement: count the pairs that MATCH, then divide by how many pairs there are.
#        In Python `True` counts as 1 in a sum, so `sum(g == h for g, h in zip(...))`
#        gives you the count directly. Divide by `len(human)`.
#      * false_accept: the grader said yes where the human said no -> `if g and not h`.
#      * false_reject: the human said yes where the grader said no -> `if h and not g`.
#      * For the two counts, `sum(1 for g, h in zip(grader, human) if <condition>)` is
#        the pattern — same shape as precision_at_k in Exercise 3.

def agreement(grader, human):
    """Compare yes/no grader verdicts against yes/no human labels (1 = yes, 0 = no).

    Returns the share they agree on, and the two errors kept apart:
      false accepts  - grader passed something the human failed  (reaches the customer)
      false rejects  - grader failed something the human passed  (costs review time)
    """
    agree = ...
    false_accept = ...
    false_reject = ...
    return agree, false_accept, false_reject


summary = []
for label, key, field, human_key in [
        ("correct — does it match reality?", "correctness", "correct", "correct"),
        ("supported — does it stick to its sources?", "faithfulness", "grounded", "grounded")]:
    g = [verdict(a["judges"][GRADER][key][field]) for a in ANSWERS]
    h = [a["human"][human_key] for a in ANSWERS]
    ag, fa, fr = agreement(g, h)
    summary.append({"property": label, "answers": len(g), "agreement": round(ag, 3),
                    "false accepts (dangerous)": fa, "false rejects (costly)": fr})
display(pd.DataFrame(summary))

### Reading the result

**The grader matched the human on 93% of *correct* verdicts and 88% of *supported* verdicts —
and across 42 answers it never once passed something a human had failed.**

Look at which column the errors landed in. False accepts: zero, both rows. Every single
disagreement is the grader being **stricter** than the person, never more lenient. For
something you are considering as a release gate, that is the asymmetry you want: the failure
mode is wasted review time, not a wrong fee reaching a customer.

But 88% is not a finding — it is an instruction to go and read the eight it got "wrong". A
percentage tells you how often you disagreed; only the disagreements tell you *why*, and
whether you actually mind.

In [ ]:
#@title The eight disagreements, in the grader's own words (run me) { display-mode: "form" }
for a in ANSWERS:
    j = a["judges"][GRADER]
    for key, field, hkey, lab in [("correctness", "correct", "correct", "correct?  "),
                                  ("faithfulness", "grounded", "grounded", "supported?")]:
        g, h = verdict(j[key][field]), a["human"][hkey]
        if g != h:
            print(f"{a['answer_id']}   {lab}   human: {'yes' if h else 'no':<3}   "
                  f"grader: {'yes' if g else 'no'}")
            print(f"   the answer : {textwrap.shorten(a['text'], 96, placeholder=' …')}")
            print(f"   grader says: {j[key]['rationale']}")
            print()

### The eight, and what they tell you

They are not eight separate problems. They are four groups, and **not one of them is the
grader misreading a fee.**

**Three are refusals** (`q21_a1`, `q22_a1`, `q25_a1`). The assistant correctly said "I cannot
find this in the published documents", was shown no passages, and the grader reasoned: no
passages, therefore nothing is supported, therefore not grounded. The human said an answer that
asserts nothing cannot be unfaithful. **Both readings are defensible, and we never wrote down
which one we meant.** That is not a model failure, it is a missing line in the specification —
and it is the most common reason evaluation numbers are wrong in practice.

**Three are terseness** (`q14_a1` — the entire answer is *"It is 24."* — plus `q11_a8` and
`q11_a9`). The number is right; the grader wanted the units and the payment frequency the
reference mentions, and without them marked the answer not correct. Another definition we never
settled: does *correct* mean "contains no error", or "contains everything the reference
contains"? Pick one, and the grader will faithfully apply whichever you picked.

**One is a missing document title** (`q20_a1`). The answer gives the overdraft rate as 10.4%;
the grader says it cannot confirm that rate applies to a *Classic* account. It is right to. The
passage came from "Classic Current Account — Schedule of Fees and Charges", but the passage
**text** never repeats the word Classic, and the grader is shown passage text, not document
titles. Nothing was wrong with the answer or the grader — the bundle *we* assembled dropped the
one word carrying the scope. **What goes into the grader's context window is a design decision,
and this is what it costs to get it slightly wrong.**

**One is the grader beating its own reference** (`q20_a2`). It flagged that the answer describes
an overdraft interest rate as "applied per instruction", when the passage says the interest
accrues on the balance daily and is charged monthly. That is a genuine contradiction, the human
label missed it, and re-reading confirms the grader. **When you calibrate a cheap grader against
human labels, some of what looks like grader error is the labels being wrong.** A disagreement
is a place to look, not a verdict about who lost.

So: six of the eight are definitions we never wrote down or context we forgot to pass, one is
the grader outperforming the human it is measured against, and **none is it waving a wrong fee
through to a customer.** That is a far more useful thing to know than "88%".

### So what do you actually do with this

The reassuring number is the least interesting thing here. What you have bought is the shape of
a claim you can defend:

> *"Our automatic grader reaches the same verdict as a human reviewer on 93% of correctness
> and 88% of groundedness judgements, over a 42-answer test set. It has never passed an answer
> a reviewer failed; every disagreement was it being stricter than us. We re-run this check
> whenever the model, the prompt or the index changes."*

Compare that with what most teams can say: *"we use an LLM to score quality."*

**A grading number with no human-agreement figure behind it is not a measurement.** It is a
number produced by a machine nobody has checked. That holds whether the agreement turns out to
be 93% or 60% — you cannot know which without doing this, and doing it here took one afternoon
of labelling and well under a cent of compute.

---

## Where we are

You started with one wrong answer and no way to tell whose problem it was. You now have five
instruments, and the incident decomposes cleanly:

1. **Ingestion** — 34 duplicate-with-different-dates passage pairs sit in the index. 🔴
2. **Retrieval** — at today's K=5, multi-fee questions get all their fees 42% of the time. 🔴
3. **Ranking** — the reranker improves ordering 14% and cannot add missing evidence. 🟢
4. **Generation** — the model was faithful to what it was shown. 🟢
5. **The customer** — was quoted €34.00 and owed €53.90. 🔴

Three of the five lights that matter are red, and **not one of them is the language model**.
The vendor conversation you can now have is specific: raise K, deduplicate the index by
effective date, and re-measure — rather than "the AI gets things wrong."

**The pattern to take away:** each instrument sees exactly one stage.

| the instrument | what it can see | what it is blind to |
|---|---|---|
| recall / coverage | whether the evidence was found | whether the evidence was any good |
| ordering (nDCG) | whether the best of it came first | whether the shortlist was missing something |
| faithfulness | whether the answer stuck to its sources | whether the sources were right |
| correctness | whether the answer matches reality | *why* it does not |
| ingestion checks | whether the index is sound | everything downstream of it |

None of them is redundant, and none of them can cover for another. A single quality score is
the average of all five, which is another way of saying it tells you nothing about any of
them.